# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to explore and process a scientific dataset defined using a Croissant schema with the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL, which ensures programmatic access to all metadata and data tables.

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Print some summary information about the dataset
print('Title:', dataset.metadata.name)
print('Description:', dataset.metadata.description)
print('License:', dataset.metadata.license)
print('Published:', dataset.metadata.datePublished)
print('Citation:', dataset.metadata.citeAs)


## 2. Data Overview
List available record sets and fields. All dataset elements are referenced using their `@id` fields.

In [ ]:
# List all available record sets (@id and name)
print('Record Sets:')
for rs in dataset.metadata.recordSets:
    print(f"  @id: {rs['@id']}  -  name: {rs.get('name', '')}")

# For each record set, list its fields (referenced by their @id and name)
record_set_ids = [rs['@id'] for rs in dataset.metadata.recordSets]
record_set_fields = {}
for rs in dataset.metadata.recordSets:
    print(f"\nRecord set: {rs['@id']}  ({rs.get('name','')})")
    if 'fields' in rs:
        record_set_fields[rs['@id']] = [fld['@id'] for fld in rs['fields']]
        for f in rs['fields']:
            print(f"    Field @id: {f['@id']}", end='')
            if 'name' in f:
                print(f"     name: {f['name']}")
            else:
                print()
    else:
        print("    No fields found in metadata.")

## 3. Data Extraction
Extract records from a given record set and load them into a pandas DataFrame. All references are by `@id`.

In [ ]:
# List of available record set @ids (from the overview above)
selected_record_set_id = None
if len(record_set_ids) > 0:
    selected_record_set_id = record_set_ids[0]  # Just take the first for this notebook
    print(f"Using record set @id: {selected_record_set_id}")
else:
    raise Exception('No record sets defined in schema.')

# Load the data from this record set
records = list(dataset.records(record_set=selected_record_set_id))
df = pd.DataFrame(records)
print(f"Loaded {len(df)} records.")
print("Fields (columns) found:")
print(df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)
Let's filter and normalize a numeric field, and group data if categorical fields exist. 
> **All references below use the `@id` as the field names.**

In [ ]:
# Choose a numeric field using its @id (list columns to select)
numeric_candidates = df.select_dtypes(include=['float64','int64']).columns.tolist()
print(f"Numeric candidates: {numeric_candidates}")
if numeric_candidates:
    numeric_field_id = numeric_candidates[0]
else:
    numeric_field_id = None
    print('No numeric fields found; skipping numeric analysis.')

if numeric_field_id is not None:
    threshold = df[numeric_field_id].mean() if pd.notnull(df[numeric_field_id].mean()) else 0
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize the numeric field
    norm_col = numeric_field_id + '_normalized'
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / (filtered_df[numeric_field_id].std())
    print(f"First 5 rows of normalized {numeric_field_id}:")
    print(filtered_df[[numeric_field_id, norm_col]].head())

    # Try to group by a categorical field
    categorical_candidates = [c for c in df.columns if df[c].dtype=='object' and c!=numeric_field_id]
    if categorical_candidates:
        group_field_id = categorical_candidates[0]
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean of {numeric_field_id} by {group_field_id} (first 5 rows):")
        print(grouped_df.head())
    else:
        print('No categorical field found to group by.')

## 5. Visualization
Let's visualize the distribution of the numeric field or the aggregation results, if fields are present.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None:
    plt.figure(figsize=(6,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.xlabel(numeric_field_id)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.show()
    
    if 'grouped_df' in locals():
        plt.figure(figsize=(8,5))
        sns.barplot(data=grouped_df, x=group_field_id, y=numeric_field_id)
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we've loaded the FAIR² dataset for mass spectrometry analysis of extracellular vesicles from human mast cells, explored its record structure, filtered and normalized a quantitative field, and visualized key statistics using the `mlcroissant` library.

- All fields and tables were referenced through their Croissant `@id` identifiers for robustness and reproducibility.
- Use the code as a template for further domain-specific analyses, for instance identifying marker proteins or comparing abundance across sample categories.

Refer to dataset documentation and field definitions in the schema for advanced analysis.
